# Task 2: Reusable Feature Pipeline

## 1. Load and Inspect Data

Load the employee, attendance, and payroll CSV files and inspect their columns, data types, and dimensions.

## 2. Data Quality Checks

Check missing values, duplicate records, employee IDs, and date ranges before combining the datasets.

## 3. Data Leakage Prevention

Use only attendance and payroll records available on or before each employee's observation date. Exclude target labels, post-outcome fields, and identifiers from model inputs.

## 4. Feature Engineering

Create attendance rate, late count, average overtime, tenure, salary versus department median, and months since last revision.

## 5. Build the Preprocessing Pipeline

Use scikit-learn to handle missing values, scale numerical features, encode categorical features, and define the final feature list.

## 6. Save and Version the Pipeline

Save the fitted pipeline using Joblib and record its version in metadata.

## 7. Test the Pipeline

Check output shape, missing values, reproducibility, and save/load functionality.

## 8. Docker Execution

Run the pipeline from the command line inside Docker.

## 9. Results and Conclusion

Summarize the final features, pipeline version, and test results.

## 1. Load and Inspect Data

Load the employee, attendance, and payroll datasets and inspect their dimensions, columns, and sample records.

In [1]:
import pandas as pd
import numpy as np

# Load the three datasets
df = pd.read_csv("/workspace/data/attrition_dataset.csv")
attendance_df = pd.read_csv("/workspace/data/attendance_monthly_features.csv")
payroll_df = pd.read_csv("/workspace/data/payroll_monthly_features.csv")

# Display dataset dimensions
print("Employee Dataset:", df.shape)
print("Attendance Dataset:", attendance_df.shape)
print("Payroll Dataset:", payroll_df.shape)

# Display column names
print("\nEmployee Columns:")
print(df.columns.tolist())

print("\nAttendance Columns:")
print(attendance_df.columns.tolist())

print("\nPayroll Columns:")
print(payroll_df.columns.tolist())

# Display first five records
display(df.head())
display(attendance_df.head())
display(payroll_df.head())

Employee Dataset: (2389, 42)
Attendance Dataset: (7339, 16)
Payroll Dataset: (7355, 23)

Employee Columns:
['row_id', 'employee_id', 'observation_date', 'department_id', 'role_id', 'job_level', 'work_location', 'work_mode', 'employment_type', 'highest_education', 'tenure_months', 'total_experience_years', 'previous_employers_count', 'commute_distance_km', 'annual_fixed_pay', 'comp_ratio', 'raises_received', 'last_raise_pct', 'months_since_last_raise', 'promotions_received', 'months_since_last_promotion', 'internal_transfers', 'last_performance_rating', 'last_engagement_score', 'manager_changes_12m', 'training_hours_12m', 'attendance_rate_3m', 'late_arrivals_3m', 'avg_overtime_hours_per_month_3m', 'avg_hours_worked_3m', 'wfh_share_3m', 'unpaid_leave_days_6m', 'paid_leave_days_6m', 'dept_attrition_rate_prior_12m', 'left_within_12m', 'voluntary_left_within_12m', 'source', 'exit_date', 'resignation_date', 'exit_reason', 'exit_interview_done', 'fnf_settlement_amount']

Attendance Columns:
[

,row_id,employee_id,observation_date,department_id,role_id,job_level,work_location,work_mode,employment_type,highest_education,...,paid_leave_days_6m,dept_attrition_rate_prior_12m,left_within_12m,voluntary_left_within_12m,source,exit_date,resignation_date,exit_reason,exit_interview_done,fnf_settlement_amount
0,1,EMP-0002,2025-09-30,D08,R047,4,Pune,Remote,Permanent,B.E. Computer Science,...,5,0.2353,0,0,master_employee_snapshot,NaN,NaN,NaN,NaN,NaN
1,2,EMP-0004,2025-09-30,D03,R017,3,Erode,Hybrid,Permanent,MCA,...,4,0.0556,0,0,master_employee_snapshot,NaN,NaN,NaN,NaN,NaN
2,3,EMP-0011,2025-09-30,D07,R039,1,Kochi,Hybrid,Permanent,M.A. Social Work,...,7,0.0000,0,0,master_employee_snapshot,NaN,NaN,NaN,NaN,NaN
3,4,EMP-0012,2025-09-30,D01,R004,2,Chennai,Onsite,Permanent,B.Tech Information Technology,...,8,0.1448,1,1,master_employee_snapshot,2026-04-23,2026-03-24,Work-Life Balance,Y,"117,098.85"
4,5,EMP-0013,2025-09-30,D04,R023,2,Chennai,Onsite,Permanent,B.E. Mechanical,...,7,0.2051,0,0,master_employee_snapshot,NaN,NaN,NaN,NaN,NaN


,attendance_feature_id,employee_id,month,department_id,job_level,scheduled_days,present_days,wfh_days,half_days_unpaid,paid_leave_days,unpaid_absence_days,longest_unpaid_streak_days,late_arrivals,avg_hours_worked,overtime_hours,attendance_rate
0,AF-00001,EMP-0002,2025-04-01,D08,4,20,20,18,0,0,0,0,1,8.50,0.0,1.0000
1,AF-00002,EMP-0002,2025-05-01,D08,4,21,20,18,1,0,0,0,2,8.71,0.0,0.9762
2,AF-00003,EMP-0002,2025-06-01,D08,4,21,19,18,1,1,0,0,1,8.25,0.0,0.9286
3,AF-00004,EMP-0002,2025-07-01,D08,4,23,23,22,0,0,0,0,1,8.63,0.0,1.0000
4,AF-00005,EMP-0002,2025-08-01,D08,4,19,19,19,0,0,0,0,1,8.62,0.0,1.0000


,payroll_item_id,run_id,employee_id,month,department_id,role_id,job_level,overtime_eligible,monthly_fixed_gross,payable_days,...,variable_pay,variable_pay_type,gross_pay,pf_employee,esi_employee,professional_tax,tds,other_deductions,total_deductions,net_pay
0,PI-000001,PR-2025-04,EMP-0002,2025-04-01,D08,R047,4,N,"144,460.00",30.0,...,"34,670.40",Performance Bonus,"179,130.40","1,800.00",0.0,200.0,"18,625.79",0.00,"20,625.79","158,504.61"
1,PI-000002,PR-2025-04,EMP-0004,2025-04-01,D03,R017,3,N,"103,938.00",30.0,...,"56,126.52",Performance Bonus,"160,064.52","1,800.00",0.0,200.0,"29,517.82",0.00,"31,517.82","128,546.70"
2,PI-000003,PR-2025-04,EMP-0006,2025-04-01,D00,R001,7,N,"479,009.00",30.0,...,0.00,NaN,"479,009.00","1,800.00",0.0,200.0,"111,100.81",0.00,"113,100.81","365,908.19"
3,PI-000004,PR-2025-04,EMP-0011,2025-04-01,D07,R039,1,Y,"24,041.00",29.5,...,"12,982.14",Performance Bonus,"36,969.21","1,134.69",0.0,200.0,0.00,0.00,"1,334.69","35,634.52"
4,PI-000005,PR-2025-04,EMP-0012,2025-04-01,D01,R004,2,N,"53,915.00",30.0,...,0.00,NaN,"53,915.00","1,800.00",0.0,200.0,0.00,0.00,"2,000.00","51,915.00"


## 2. Data Quality Checks and Cleaning

Check and handle duplicate payroll records and prepare the date and numeric columns needed for feature engineering. These steps prevent payroll amounts from being counted twice and ensure that only valid historical records are used.

In [2]:
# Convert date columns to datetime format
df["observation_date"] = pd.to_datetime(df["observation_date"])

attendance_df["month"] = pd.to_datetime(attendance_df["month"])
payroll_df["month"] = pd.to_datetime(payroll_df["month"])

# Check repeated payroll employee-month records
payroll_duplicates = payroll_df.duplicated(
    subset=["employee_id", "month"],
    keep=False
)

print("Repeated payroll employee-month rows:", payroll_duplicates.sum())

# Display repeated payroll records for inspection
display(
    payroll_df[payroll_duplicates].sort_values(
        ["employee_id", "month"]
    ).head(10)
)

Repeated payroll employee-month rows: 30


,payroll_item_id,run_id,employee_id,month,department_id,role_id,job_level,overtime_eligible,monthly_fixed_gross,payable_days,...,variable_pay,variable_pay_type,gross_pay,pf_employee,esi_employee,professional_tax,tds,other_deductions,total_deductions,net_pay
3975,PI-003970,PR-2026-02,EMP-0012,2026-02-01,D01,R004,2,N,"53,915.00",28.0,...,0.00,NaN,"53,915.00","1,800.00",0.0,200.0,0.00,0.00,"2,000.00","51,915.00"
3976,PI-D00007,PR-2026-02,EMP-0012,2026-02-01,D01,R004,2,N,"53,915.00",28.0,...,0.00,NaN,"53,915.00","1,800.00",0.0,200.0,0.00,0.00,"2,000.00","51,915.00"
5673,PI-005661,PR-2026-06,EMP-0092,2026-06-01,D08,R045,2,Y,"73,903.00",30.0,...,0.00,NaN,"73,903.00","1,800.00",0.0,200.0,0.00,0.00,"2,000.00","71,903.00"
5674,PI-D00015,PR-2026-06,EMP-0092,2026-06-01,D08,R045,2,Y,"73,903.00",30.0,...,0.00,NaN,"73,903.00","1,800.00",0.0,200.0,0.00,0.00,"2,000.00","71,903.00"
2416,PI-002412,PR-2025-10,EMP-0112,2025-10-01,D01,R009,6,N,"332,694.00",31.0,...,"10,000.00",Festival Bonus,"342,694.00","1,800.00",0.0,200.0,"68,570.53",0.00,"70,570.53","272,123.47"
2417,PI-D00002,PR-2025-10,EMP-0112,2025-10-01,D01,R009,6,N,"332,694.00",31.0,...,"10,000.00",Festival Bonus,"342,694.00","1,800.00",0.0,200.0,"68,570.53",0.00,"70,570.53","272,123.47"
4058,PI-004052,PR-2026-02,EMP-0179,2026-02-01,D06,R035,3,N,"88,641.00",28.0,...,0.00,NaN,"88,641.00","1,800.00",0.0,200.0,0.00,0.00,"2,000.00","86,641.00"
4059,PI-D00001,PR-2026-02,EMP-0179,2026-02-01,D06,R035,3,N,"88,641.00",28.0,...,0.00,NaN,"88,641.00","1,800.00",0.0,200.0,0.00,0.00,"2,000.00","86,641.00"
496,PI-000497,PR-2025-05,EMP-0218,2025-05-01,D02,R013,4,N,"178,808.00",31.0,...,0.00,NaN,"178,808.00","1,800.00",0.0,200.0,"18,865.08",0.00,"20,865.08","157,942.92"
497,PI-D00008,PR-2025-05,EMP-0218,2025-05-01,D02,R013,4,N,"178,808.00",31.0,...,0.00,NaN,"178,808.00","1,800.00",0.0,200.0,"18,865.08",0.00,"20,865.08","157,942.92"


In [3]:
# Remove duplicate payroll records, ignoring the unique item ID
payroll_columns = [
    column for column in payroll_df.columns
    if column != "payroll_item_id"
]

payroll_df = payroll_df.drop_duplicates(
    subset=payroll_columns
).copy()

print("Payroll rows after removing duplicates:", len(payroll_df))

# Convert payroll amount columns to numeric values
payroll_amount_columns = [
    "monthly_fixed_gross",
    "overtime_pay",
    "variable_pay",
    "gross_pay",
    "pf_employee",
    "esi_employee",
    "professional_tax",
    "tds",
    "other_deductions",
    "total_deductions",
    "net_pay"
]

for column in payroll_amount_columns:
    payroll_df[column] = pd.to_numeric(
        payroll_df[column].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

print("Payroll cleaning completed.")
print("Remaining duplicate payroll records:", payroll_df.duplicated(
    subset=payroll_columns
).sum())

Payroll rows after removing duplicates: 7340
Payroll cleaning completed.
Remaining duplicate payroll records: 0


## 3. Feature Engineering

Create attendance rate, late count, average overtime, tenure, salary versus department median, and months since last revision. Only monthly records available on or before each employee's observation date will be used.

In [4]:
import sys
sys.path.insert(0, "/workspace")

from src.features.build_features import build_features

features_df = build_features(
    df,
    attendance_df,
    payroll_df
)

print("Feature engineering completed.")
print("Output shape:", features_df.shape)

display(features_df.head())

Feature engineering completed.
Output shape: (2389, 48)


,row_id,employee_id,observation_date,department_id,role_id,job_level,work_location,work_mode,employment_type,highest_education,...,resignation_date,exit_reason,exit_interview_done,fnf_settlement_amount,attendance_rate,late_count,average_overtime,average_monthly_salary,tenure,months_since_last_revision
0,1,EMP-0002,2025-09-30,D08,R047,4,Pune,Remote,Permanent,B.E. Computer Science,...,NaN,NaN,NaN,NaN,0.946250,7.0,0.000000,144460.0,164,5
1,2,EMP-0004,2025-09-30,D03,R017,3,Erode,Hybrid,Permanent,MCA,...,NaN,NaN,NaN,NaN,0.960167,4.0,0.000000,140316.3,162,5
2,3,EMP-0011,2025-09-30,D07,R039,1,Kochi,Hybrid,Permanent,M.A. Social Work,...,NaN,NaN,NaN,NaN,0.937600,4.0,3.416667,24041.0,154,5
3,4,EMP-0012,2025-09-30,D01,R004,2,Chennai,Onsite,Permanent,B.Tech Information Technology,...,2026-03-24,Work-Life Balance,Y,"117,098.85",0.938933,21.0,0.000000,53915.0,152,5
4,5,EMP-0013,2025-09-30,D04,R023,2,Chennai,Onsite,Permanent,B.E. Mechanical,...,NaN,NaN,NaN,NaN,0.946967,14.0,0.000000,48514.0,152,5


In [5]:

from src.features.build_features import build_features

# Use the reusable feature engineering function
features_df = build_features(
    df,
    attendance_df,
    payroll_df
)

print("Feature engineering completed.")
print("Output shape:", features_df.shape)

display(
    features_df[
        [
            "employee_id",
            "attendance_rate",
            "late_count",
            "average_overtime",
            "tenure",
            "annual_fixed_pay",
            "months_since_last_revision",
        ]
    ].head()
)


Feature engineering completed.
Output shape: (2389, 48)


,employee_id,attendance_rate,late_count,average_overtime,tenure,annual_fixed_pay,months_since_last_revision
0,EMP-0002,0.946250,7.0,0.000000,164,1733520,5
1,EMP-0004,0.960167,4.0,0.000000,162,1247256,5
2,EMP-0011,0.937600,4.0,3.416667,154,288492,5
3,EMP-0012,0.938933,21.0,0.000000,152,646980,5
4,EMP-0013,0.946967,14.0,0.000000,152,582168,5


In [6]:
# Check missing values in the engineered features
feature_columns = [
    "attendance_rate",
    "late_count",
    "average_overtime",
    "tenure",
    "annual_fixed_pay",
    "months_since_last_revision"
]

print(features_df[feature_columns].isnull().sum())

attendance_rate               2000
late_count                       0
average_overtime              2000
tenure                           0
annual_fixed_pay                 0
months_since_last_revision       0
dtype: int64


In [7]:
# Check how many employees have monthly records
print("Employee records:", len(df))
print("Employees with attendance history:", attendance_df["employee_id"].nunique())
print("Employees with payroll history:", payroll_df["employee_id"].nunique())

print(
    "Employees with attendance features:",
    features_df["attendance_rate"].notna().sum()
)

print(
    "Employees without attendance features:",
    features_df["attendance_rate"].isna().sum()
)

Employee records: 2389
Employees with attendance history: 528
Employees with payroll history: 528
Employees with attendance features: 389
Employees without attendance features: 2000


## 4. Build the Preprocessing Pipeline

Create a reusable scikit-learn pipeline to handle missing values, scale numerical features, and encode categorical features consistently.

In [8]:
from src.features.pipeline import (
    create_preprocessing_pipeline,
    FEATURE_VERSION,
    FINAL_FEATURES
)

preprocessing = create_preprocessing_pipeline()

print("Pipeline version:", FEATURE_VERSION)
print("Number of input features:", len(FINAL_FEATURES))
print("Pipeline created successfully.")

Pipeline version: features-1.0.0
Number of input features: 11
Pipeline created successfully.


In [9]:
import sys

sys.path.insert(0, "/workspace")

from src.features.pipeline import (
    create_preprocessing_pipeline,
    FEATURE_VERSION,
    FINAL_FEATURES
)

preprocessing = create_preprocessing_pipeline()

print("Pipeline version:", FEATURE_VERSION)
print("Number of input features:", len(FINAL_FEATURES))
print("Pipeline created successfully.")

Pipeline version: features-1.0.0
Number of input features: 11
Pipeline created successfully.


## 5. Fit and Save the Pipeline

Fit the preprocessing pipeline on training data only, then save it with Joblib and record its version in metadata.

In [10]:
from sklearn.model_selection import train_test_split
from src.features.pipeline import (
    create_preprocessing_pipeline,
    save_pipeline,
    FINAL_FEATURES
)

# Exclude identifiers and outcome labels from model inputs
X = features_df[FINAL_FEATURES].copy()
y = features_df["left_within_12m"].copy()

# Split before fitting preprocessing steps
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Fit only on training data
preprocessing = create_preprocessing_pipeline()
X_train_transformed = preprocessing.fit_transform(X_train)

# Save the pipeline fitted on training data
save_pipeline(
    preprocessing,
    output_dir="/workspace/models"
)

print("Training rows:", X_train.shape[0])
print("Transformed training shape:", X_train_transformed.shape)
print("Pipeline fitted and saved.")

Pipeline and metadata saved successfully.
Training rows: 1911
Transformed training shape: (1911, 85)
Pipeline fitted and saved.


In [11]:
import src.features.pipeline as pipeline_module

save_pipeline = pipeline_module.save_pipeline

print("save_pipeline imported successfully.")

save_pipeline imported successfully.


In [12]:
from src.features.build_features import build_features

features_df = build_features(df, attendance_df, payroll_df)

print("Output shape:", features_df.shape)
print("Unique employee snapshots:", features_df["employee_id"].nunique())
print("\nEngineered features:")
print(features_df[
    [
        "attendance_rate",
        "late_count",
        "average_overtime",
        "tenure",
        "annual_fixed_pay",
        "months_since_last_revision",
    ]
].head())

print("\nMissing values in engineered features:")

feature_columns = [
    "attendance_rate",
    "late_count",
    "average_overtime",
    "tenure",
    "annual_fixed_pay",
    "months_since_last_revision",
]

print(features_df[feature_columns].isnull().sum())

Output shape: (2389, 48)
Unique employee snapshots: 2383

Engineered features:
   attendance_rate  late_count  average_overtime  tenure  annual_fixed_pay  \
0         0.946250         7.0          0.000000     164           1733520   
1         0.960167         4.0          0.000000     162           1247256   
2         0.937600         4.0          3.416667     154            288492   
3         0.938933        21.0          0.000000     152            646980   
4         0.946967        14.0          0.000000     152            582168   

   months_since_last_revision  
0                           5  
1                           5  
2                           5  
3                           5  
4                           5  

Missing values in engineered features:
attendance_rate               2000
late_count                       0
average_overtime              2000
tenure                           0
annual_fixed_pay                 0
months_since_last_revision       0
dtype: int

In [13]:

# Save the pipeline fitted on training data in Cell 15.
save_pipeline(
    preprocessing,
    output_dir="/workspace/models"
)

print("Training-fitted pipeline saved successfully.")


Pipeline and metadata saved successfully.
Training-fitted pipeline saved successfully.


In [14]:
from pathlib import Path
import json
import joblib

model_dir = Path("/workspace/models")

print("Pipeline exists:", (model_dir / "feature_pipeline.joblib").exists())
print("Metadata exists:", (model_dir / "metadata.json").exists())

with open(model_dir / "metadata.json", "r") as file:
    metadata = json.load(file)

print("\nPipeline version:", metadata["version"])
print("Numeric features:", len(metadata["numeric_features"]))
print("Categorical features:", len(metadata["categorical_features"]))

loaded_pipeline = joblib.load(model_dir / "feature_pipeline.joblib")
print("\nSaved pipeline loaded successfully.")

Pipeline exists: True
Metadata exists: True

Pipeline version: features-1.0.0
Numeric features: 6
Categorical features: 5

Saved pipeline loaded successfully.


In [15]:
import joblib
import numpy as np
from scipy import sparse

loaded_pipeline = joblib.load(
    "/workspace/models/feature_pipeline.joblib"
)

X = features_df[FINAL_FEATURES].copy()
transformed = loaded_pipeline.transform(X)

values = transformed.data if sparse.issparse(transformed) else np.asarray(transformed)

print("Input rows:", X.shape[0])
print("Transformed shape:", transformed.shape)
print("Contains missing values:", np.isnan(values).any())

Input rows: 2389
Transformed shape: (2389, 85)
Contains missing values: False
